In [1]:
from sqlalchemy import create_engine, text
import pandas as pd

# Windows автентикация
connection_string = (
    "mssql+pyodbc://@localhost/ProductsDB"
    "?driver=ODBC+Driver+17+for+SQL+Server" # Трябва да е инсталиран
    "&trusted_connection=yes"
    "&TrustServerCertificate=yes")

engine = create_engine(connection_string)

In [10]:
# SQL Server автентикация
connection_string = (
    "mssql+pyodbc://sa:@localhost/ProductsDB" # sa - потребителско име; парола след двоеточието (ако има)
    "?driver=ODBC+Driver+17+for+SQL+Server"
    "&TrustServerCertificate=yes")

engine = create_engine(connection_string)

In [11]:
query_sel = """
SELECT ProductName, Stock
FROM Products
WHERE Stock < 100
"""

df = pd.read_sql(query_sel, engine)
print(df)

       ProductName  Stock
0           ябълки   75.0
1               xx   90.0
2          картофи   55.0
3             чипс   20.0
4  царевичен снакс   24.0
5         макарони   34.0
6           еклери   52.0
7              bbb   12.0


In [3]:
query_del = """DELETE FROM Products
WHERE ProductName = 'cccc'
"""

with engine.begin() as connection:
    result = connection.execute(text(query_del))
    print("Изтрити редове:", result.rowcount)
    
df = pd.read_sql(query_sel, engine)
print(df)

Изтрити редове: 0
       ProductName  Stock
0           ябълки   75.0
1               xx   90.0
2          картофи   55.0
3             чипс   20.0
4  царевичен снакс   24.0
5         макарони   34.0
6           еклери   52.0
7              bbb   12.0


In [4]:
query_with = """WITH Duplicates AS (
    SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY ProductID, Stock
            ORDER BY (SELECT NULL)
        ) AS rn
    FROM Products
)
DELETE FROM Duplicates
WHERE rn > 1 
"""

with engine.begin() as connection:
    result = connection.execute(text(query_with))
    print("Изтрити редове:", result.rowcount)
    
df = pd.read_sql(query_sel, engine)
print(df)

Изтрити редове: 0
       ProductName  Stock
0           ябълки   75.0
1               xx   90.0
2          картофи   55.0
3             чипс   20.0
4  царевичен снакс   24.0
5         макарони   34.0
6           еклери   52.0
7              bbb   12.0


In [5]:
df_filled = df.copy()